In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import numpy as np

# MLPの定義 (通常のNNとして定義)
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_units, activations):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_dim

        # 指定された hidden_units と activations に基づいて層を構築
        for hidden_dim, activation in zip(hidden_units, activations):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(self.get_activation(activation))
            in_dim = hidden_dim

        # 最後の出力層
        layers.append(nn.Linear(in_dim, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

    @staticmethod
    def get_activation(activation):
        # 活性化関数の設定
        if activation == "relu":
            return nn.ReLU()
        elif activation == "tanh":
            return nn.Tanh()
        elif activation == "sigmoid":
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation function: {activation}")

# 真の関数 (ground truth)
def true_function(x):
    return 3 * np.sin(x) + x

# ラプラス近似のためのヘッセ行列近似を計算する関数
def hessian_diag(model, loss):
    # ヘッセ行列の対角成分を計算
    loss.backward(create_graph=True)
    diag_hessian = []
    for param in model.parameters():
        diag_hessian.append(param.grad.pow(2).detach())
    return diag_hessian

# ラプラス近似に基づく予測
def laplace_approximation_predict(model, X_test, diag_hessian, n_samples=100):
    sampled_preds = []

    for _ in range(n_samples):
        perturbed_model = MLP(**model_settings)
        for p_perturbed, p_orig, h_diag in zip(perturbed_model.parameters(), model.parameters(), diag_hessian):
            # ラプラス近似に基づいてガウス分布から重みをサンプリング
            std = torch.sqrt(h_diag)
            perturbed_model_param = p_orig + torch.randn_like(p_orig) * std
            p_perturbed.data.copy_(perturbed_model_param)
        
        sampled_preds.append(perturbed_model(X_test).detach())

    # すべてのサンプルから平均と標準偏差を計算
    preds = torch.stack(sampled_preds)
    mean_preds = preds.mean(0)
    std_preds = preds.std(0)
    
    return mean_preds, std_preds

# モデルのトレーニングと予測
def train_and_predict_with_laplace(model_settings, num_epochs=1000):
    # データ準備
    torch.manual_seed(42)
    X_train = torch.randn(100, 1)  # 訓練データ
    y_train = true_function(X_train) + torch.randn(100, 1)  # ノイズを加えた出力

    model = MLP(**model_settings)  # モデルのインスタンス化
    criterion = nn.MSELoss()  # 損失関数
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # モデルのトレーニング
    for epoch in range(num_epochs):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    # ラプラス近似のためにヘッセ行列の対角成分を計算
    model.eval()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    diag_hessian = hessian_diag(model, loss)

    # ラプラス近似に基づく予測
    X_test = torch.linspace(-3, 3, 100).unsqueeze(1)  # テストデータ
    mean_preds, std_preds = laplace_approximation_predict(model, X_test, diag_hessian)

    return X_train, y_train, X_test, mean_preds, std_preds


# プロット
def plot_results(X_train, y_train, X_test, mean_preds, std_preds, n_sigma=2):
    # データの変換
    X_train_np, y_train_np = X_train.numpy(), y_train.numpy()
    X_test_np, mean_preds_np = X_test.numpy(), mean_preds.numpy()
    y_true_np = true_function(X_test_np)
    std_preds_np = std_preds.numpy()

    # Plotlyによる可視化
    fig = go.Figure()

    # 訓練データ点
    fig.add_trace(go.Scatter(
        x=X_train_np.flatten(),
        y=y_train_np.flatten(),
        mode='markers',
        name='Training Data',
        marker=dict(color='blue', size=6)
    ))

    # 真の関数の軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=y_true_np.flatten(),
        mode='lines',
        name='True Function',
        line=dict(color='green', width=2)
    ))

    # NN の予測の平均軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=mean_preds_np.flatten(),
        mode='lines',
        name='NN Prediction (Mean)',
        line=dict(color='red', width=2)
    ))

    # 標準偏差の範囲をプロット (不確実性を表示)
    fig.add_trace(go.Scatter(
        x=np.concatenate([X_test_np.flatten(), X_test_np.flatten()[::-1]]),
        y=np.concatenate([mean_preds_np.flatten() - n_sigma * std_preds_np.flatten(), 
                          (mean_preds_np.flatten() + n_sigma * std_preds_np.flatten())[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.2)',  # 塗りつぶしの色
        line=dict(color='rgba(255, 0, 0, 0)'),  # 標準偏差の範囲の線の色を透明に
        hoverinfo="skip",
        showlegend=False
    ))

    # グラフのレイアウト設定
    fig.update_layout(
        title="BNN with Laplace Approximation",
        xaxis_title="X",
        yaxis_title="y",
        width=800,
        height=600
    )

    fig.show()


if __name__ == "__main__":
    # モデルの設定
    model_settings = {
        "input_dim": 1,
        "output_dim": 1,
        "hidden_units": [128, 128, 128],
        "activations": ["tanh", "tanh", "tanh"]
    }

    # トレーニングとラプラス近似での予測
    X_train, y_train, X_test, mean_preds, std_preds = train_and_predict_with_laplace(model_settings)

    # 結果の可視化
    plot_results(X_train, y_train, X_test, mean_preds, std_preds, n_sigma=5)




Epoch [100/1000], Loss: 0.7119
Epoch [200/1000], Loss: 0.6351
Epoch [300/1000], Loss: 0.5517
Epoch [400/1000], Loss: 0.4861
Epoch [500/1000], Loss: 0.4168
Epoch [600/1000], Loss: 0.4123
Epoch [700/1000], Loss: 0.3845
Epoch [800/1000], Loss: 0.3256
Epoch [900/1000], Loss: 0.2986
Epoch [1000/1000], Loss: 0.2550


In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import numpy as np

# MLPの定義 (通常のNNとして定義)
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_units, activations):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_dim

        # 指定された hidden_units と activations に基づいて層を構築
        for hidden_dim, activation in zip(hidden_units, activations):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(self.get_activation(activation))
            in_dim = hidden_dim

        # 最後の出力層
        layers.append(nn.Linear(in_dim, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

    @staticmethod
    def get_activation(activation):
        # 活性化関数の設定
        if activation == "relu":
            return nn.ReLU()
        elif activation == "tanh":
            return nn.Tanh()
        elif activation == "sigmoid":
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation function: {activation}")

# 1D データ生成（アウトライア含む）
def generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.0):
    # x を 0 から 2π の範囲で生成
    x = np.linspace(0, 2 * np.pi, n)
    
    # 真の関数は sin 関数
    y_true = np.sin(2 * x)
    
    # y にガウスノイズを加える
    y = y_true + np.random.normal(0, noise_std, x.shape[0])
    
    # アウトライアを挿入
    n_outliers = int(n * outlier_ratio)
    outlier_indices = np.random.choice(n, n_outliers, replace=False)
    y[outlier_indices] = 5 + np.random.uniform(-0.1, 0.1, n_outliers)
    
    return x, y, y_true, outlier_indices

# ラプラス近似のためのヘッセ行列近似を計算する関数
def hessian_diag(model, loss):
    # ヘッセ行列の対角成分を計算
    loss.backward(create_graph=True)
    diag_hessian = []
    for param in model.parameters():
        diag_hessian.append(param.grad.pow(2).detach())
    return diag_hessian

# ラプラス近似に基づく予測
def laplace_approximation_predict(model, X_test, diag_hessian, n_samples=100):
    sampled_preds = []

    for _ in range(n_samples):
        perturbed_model = MLP(**model_settings)
        for p_perturbed, p_orig, h_diag in zip(perturbed_model.parameters(), model.parameters(), diag_hessian):
            # ラプラス近似に基づいてガウス分布から重みをサンプリング
            std = torch.sqrt(h_diag)
            perturbed_model_param = p_orig + torch.randn_like(p_orig) * std
            p_perturbed.data.copy_(perturbed_model_param)
        
        sampled_preds.append(perturbed_model(X_test).detach())

    # すべてのサンプルから平均と標準偏差を計算
    preds = torch.stack(sampled_preds)
    mean_preds = preds.mean(0)
    std_preds = preds.std(0)
    
    return mean_preds, std_preds


def train_and_predict_with_laplace(model_settings, x_train, y_train, num_epochs=1000, l2_coeff=1e-4):
    X_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

    model = MLP(**model_settings)  # モデルのインスタンス化
    criterion = nn.MSELoss()  # 基本の損失関数
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # モデルのトレーニング
    for epoch in range(num_epochs):
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    # ラプラス近似のためにヘッセ行列の対角成分を計算
    model.eval()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    diag_hessian = hessian_diag(model, loss)

    # ラプラス近似に基づく予測
    X_test = torch.linspace(0, 2 * np.pi, 100).unsqueeze(1)  # テストデータ
    mean_preds, std_preds = laplace_approximation_predict(model, X_test, diag_hessian)

    return X_train, y_train, X_test, mean_preds, std_preds

# プロット
def plot_results(x_train, y_train, y_true, outlier_indices, X_test, mean_preds, std_preds):
    # データの変換
    mean_preds_np = mean_preds.numpy()
    std_preds_np = std_preds.numpy()
    X_test_np = X_test.numpy()
    y_true_test = np.sin(2 * X_test_np.flatten())

    # Plotlyによる可視化
    fig = go.Figure()

    # 訓練データ点
    fig.add_trace(go.Scatter(
        x=x_train,
        y=y_train,
        mode='markers',
        name='Training Data',
        marker=dict(color='blue', size=6)
    ))

    # アウトライアを強調表示
    fig.add_trace(go.Scatter(
        x=x_train[outlier_indices],
        y=y_train[outlier_indices],
        mode='markers',
        name='Outliers',
        marker=dict(color='red', size=8, symbol='x')
    ))

    # 真の関数の軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=y_true_test,
        mode='lines',
        name='True Function',
        line=dict(color='green', width=2)
    ))

    # NN の予測の平均軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=mean_preds_np.flatten(),
        mode='lines',
        name='NN Prediction (Mean)',
        line=dict(color='red', width=2)
    ))

    # 標準偏差の範囲をプロット (不確実性を表示)
    fig.add_trace(go.Scatter(
        x=np.concatenate([X_test_np.flatten(), X_test_np.flatten()[::-1]]),
        y=np.concatenate([mean_preds_np.flatten() - std_preds_np.flatten(), 
                          (mean_preds_np.flatten() + std_preds_np.flatten())[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.2)',  # 塗りつぶしの色
        line=dict(color='rgba(255, 0, 0, 0)'),  # 標準偏差の範囲の線の色を透明に
        hoverinfo="skip",
        showlegend=False
    ))

    # グラフのレイアウト設定
    fig.update_layout(
        title="BNN with Laplace Approximation (with Outliers)",
        xaxis_title="X",
        yaxis_title="y",
        width=800,
        height=600
    )

    fig.show()

if __name__ == "__main__":
    # データ生成
    x_train, y_train, y_true, outlier_indices = generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05)

    # モデルの設定
    model_settings = {
        "input_dim": 1,
        "output_dim": 1,
        "hidden_units": [10, 20],
        "activations": ["relu", "tanh"]
    }

    # トレーニングとラプラス近似での予測
    X_train, y_train_tensor, X_test, mean_preds, std_preds = train_and_predict_with_laplace(
        model_settings, x_train, y_train)

    # 結果の可視化
    plot_results(x_train, y_train, y_true, outlier_indices, X_test, mean_preds, std_preds)


Epoch [100/1000], Loss: 1.3072
Epoch [200/1000], Loss: 1.2204
Epoch [300/1000], Loss: 1.2088
Epoch [400/1000], Loss: 1.1652
Epoch [500/1000], Loss: 1.0884
Epoch [600/1000], Loss: 1.0162
Epoch [700/1000], Loss: 0.9819
Epoch [800/1000], Loss: 0.9514
Epoch [900/1000], Loss: 0.9152
Epoch [1000/1000], Loss: 0.8638


# L2正則化あり

In [61]:
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import numpy as np

# MLPの定義 (通常のNNとして定義)
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_units, activations):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_dim

        # 指定された hidden_units と activations に基づいて層を構築
        for hidden_dim, activation in zip(hidden_units, activations):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(self.get_activation(activation))
            in_dim = hidden_dim

        # 最後の出力層
        layers.append(nn.Linear(in_dim, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

    @staticmethod
    def get_activation(activation):
        # 活性化関数の設定
        if activation == "relu":
            return nn.ReLU()
        elif activation == "tanh":
            return nn.Tanh()
        elif activation == "sigmoid":
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation function: {activation}")

# 真の関数 (ground truth)
def true_function(x):
    return 3 * np.sin(x) + x

# ラプラス近似のためのヘッセ行列近似を計算する関数
def hessian_diag(model, loss):
    # ヘッセ行列の対角成分を計算
    loss.backward(create_graph=True)
    diag_hessian = []
    for param in model.parameters():
        diag_hessian.append(param.grad.pow(2).detach())
    return diag_hessian

# ラプラス近似に基づく予測
def laplace_approximation_predict(model, X_test, diag_hessian, n_samples=100):
    sampled_preds = []

    for _ in range(n_samples):
        perturbed_model = MLP(**model_settings)
        for p_perturbed, p_orig, h_diag in zip(perturbed_model.parameters(), model.parameters(), diag_hessian):
            # ラプラス近似に基づいてガウス分布から重みをサンプリング
            std = torch.sqrt(h_diag)
            perturbed_model_param = p_orig + torch.randn_like(p_orig) * std
            p_perturbed.data.copy_(perturbed_model_param)
        
        sampled_preds.append(perturbed_model(X_test).detach())

    # すべてのサンプルから平均と標準偏差を計算
    preds = torch.stack(sampled_preds)
    mean_preds = preds.mean(0)
    std_preds = preds.std(0)
    
    return mean_preds, std_preds

# モデルのトレーニングと予測
def train_and_predict_with_laplace(model_settings, num_epochs=1000, l2_coeff=1e-4):
    # データ準備
    torch.manual_seed(42)
    X_train = torch.randn(100, 1)  # 訓練データ
    y_train = true_function(X_train) + torch.randn(100, 1)  # ノイズを加えた出力

    model = MLP(**model_settings)  # モデルのインスタンス化
    criterion = nn.MSELoss()  # 損失関数
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # モデルのトレーニング
    for epoch in range(num_epochs):
        model.train()
        outputs = model(X_train)
        mse_loss = criterion(outputs, y_train)

        # L2 正則化の計算
        l2_reg = torch.tensor(0.0)
        for param in model.parameters():
            l2_reg += torch.norm(param, 2)

        # 総損失 (MSE + L2 正則化)
        total_loss = mse_loss + l2_coeff * l2_reg

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss.item():.4f}')

    # ラプラス近似のためにヘッセ行列の対角成分を計算
    model.eval()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    diag_hessian = hessian_diag(model, loss)

    # ラプラス近似に基づく予測
    X_test = torch.linspace(-3, 3, 100).unsqueeze(1)  # テストデータ
    mean_preds, std_preds = laplace_approximation_predict(model, X_test, diag_hessian)

    return X_train, y_train, X_test, mean_preds, std_preds


# プロット
def plot_results(X_train, y_train, X_test, mean_preds, std_preds, n_sigma=2):
    # データの変換
    X_train_np, y_train_np = X_train.numpy(), y_train.numpy()
    X_test_np, mean_preds_np = X_test.numpy(), mean_preds.numpy()
    y_true_np = true_function(X_test_np)
    std_preds_np = std_preds.numpy()

    # Plotlyによる可視化
    fig = go.Figure()

    # 訓練データ点
    fig.add_trace(go.Scatter(
        x=X_train_np.flatten(),
        y=y_train_np.flatten(),
        mode='markers',
        name='Training Data',
        marker=dict(color='blue', size=6)
    ))

    # 真の関数の軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=y_true_np.flatten(),
        mode='lines',
        name='True Function',
        line=dict(color='green', width=2)
    ))

    # NN の予測の平均軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=mean_preds_np.flatten(),
        mode='lines',
        name='NN Prediction (Mean)',
        line=dict(color='red', width=2)
    ))

    # 標準偏差の範囲をプロット (不確実性を表示)
    fig.add_trace(go.Scatter(
        x=np.concatenate([X_test_np.flatten(), X_test_np.flatten()[::-1]]),
        y=np.concatenate([mean_preds_np.flatten() - n_sigma * std_preds_np.flatten(), 
                          (mean_preds_np.flatten() + n_sigma * std_preds_np.flatten())[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.2)',  # 塗りつぶしの色
        line=dict(color='rgba(255, 0, 0, 0)'),  # 標準偏差の範囲の線の色を透明に
        hoverinfo="skip",
        showlegend=False
    ))

    # グラフのレイアウト設定
    fig.update_layout(
        title="BNN with Laplace Approximation",
        xaxis_title="X",
        yaxis_title="y",
        width=800,
        height=600
    )

    fig.show()


if __name__ == "__main__":
    # モデルの設定
    model_settings = {
        "input_dim": 1,
        "output_dim": 1,
        "hidden_units": [128, 128, 128],
        "activations": ["tanh", "tanh", "tanh"]
    }

    # トレーニングとラプラス近似での予測
    X_train, y_train, X_test, mean_preds, std_preds = train_and_predict_with_laplace(model_settings, l2_coeff=1e-2)

    # 結果の可視化
    plot_results(X_train, y_train, X_test, mean_preds, std_preds, n_sigma=5)


Epoch [100/1000], Loss: 0.9926
Epoch [200/1000], Loss: 0.9249
Epoch [300/1000], Loss: 0.8675
Epoch [400/1000], Loss: 0.8397
Epoch [500/1000], Loss: 0.8164
Epoch [600/1000], Loss: 0.7957
Epoch [700/1000], Loss: 0.7739
Epoch [800/1000], Loss: 0.7636
Epoch [900/1000], Loss: 0.7602
Epoch [1000/1000], Loss: 0.7613


In [117]:
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
import numpy as np

# MLPの定義 (通常のNNとして定義)
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_units, activations):
        super(MLP, self).__init__()
        layers = []
        in_dim = input_dim

        # 指定された hidden_units と activations に基づいて層を構築
        for hidden_dim, activation in zip(hidden_units, activations):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(self.get_activation(activation))
            in_dim = hidden_dim

        # 最後の出力層
        layers.append(nn.Linear(in_dim, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

    @staticmethod
    def get_activation(activation):
        # 活性化関数の設定
        if activation == "relu":
            return nn.ReLU()
        elif activation == "tanh":
            return nn.Tanh()
        elif activation == "sigmoid":
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation function: {activation}")

# 1D データ生成（アウトライア含む）
def generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.0):
    # x を 0 から 2π の範囲で生成
    x = np.linspace(0, 2 * np.pi, n)
    
    # 真の関数は sin 関数
    y_true = np.sin(2 * x)
    
    # y にガウスノイズを加える
    y = y_true + np.random.normal(0, noise_std, x.shape[0])
    
    # アウトライアを挿入
    n_outliers = int(n * outlier_ratio)
    outlier_indices = np.random.choice(n, n_outliers, replace=False)
    y[outlier_indices] = 5 + np.random.uniform(-0.1, 0.1, n_outliers)
    
    return x, y, y_true, outlier_indices

# ラプラス近似のためのヘッセ行列近似を計算する関数
def hessian_diag(model, loss):
    # ヘッセ行列の対角成分を計算
    loss.backward(create_graph=True)
    diag_hessian = []
    for param in model.parameters():
        diag_hessian.append(param.grad.pow(2).detach())
    return diag_hessian

# ラプラス近似に基づく予測
def laplace_approximation_predict(model, X_test, diag_hessian, n_samples=100):
    sampled_preds = []

    for _ in range(n_samples):
        perturbed_model = MLP(**model_settings)
        for p_perturbed, p_orig, h_diag in zip(perturbed_model.parameters(), model.parameters(), diag_hessian):
            # ラプラス近似に基づいてガウス分布から重みをサンプリング
            std = torch.sqrt(h_diag)
            perturbed_model_param = p_orig + torch.randn_like(p_orig) * std
            p_perturbed.data.copy_(perturbed_model_param)
        
        sampled_preds.append(perturbed_model(X_test).detach())

    # すべてのサンプルから平均と標準偏差を計算
    preds = torch.stack(sampled_preds)
    mean_preds = preds.mean(0)
    std_preds = preds.std(0)
    
    return mean_preds, std_preds


def train_and_predict_with_laplace(model_settings, x_train, y_train, num_epochs=1000, l2_coeff=1e-4):
    X_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

    model = MLP(**model_settings)  # モデルのインスタンス化
    criterion = nn.MSELoss()  # 基本の損失関数
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # モデルのトレーニング
    for epoch in range(num_epochs):
        model.train()
        outputs = model(X_train)
        mse_loss = criterion(outputs, y_train)

        # L2 正則化を損失に追加
        l2_reg = torch.tensor(0.0)
        for param in model.parameters():
            l2_reg += torch.norm(param, 2)
        total_loss = mse_loss + l2_coeff * l2_reg

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss.item():.4f}')

    # ラプラス近似のためにヘッセ行列の対角成分を計算
    model.eval()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    diag_hessian = hessian_diag(model, loss)

    # ラプラス近似に基づく予測
    X_test = torch.linspace(0, 2 * np.pi, 100).unsqueeze(1)  # テストデータ
    mean_preds, std_preds = laplace_approximation_predict(model, X_test, diag_hessian)

    return X_train, y_train, X_test, mean_preds, std_preds

# プロット
def plot_results(x_train, y_train, y_true, outlier_indices, X_test, mean_preds, std_preds):
    # データの変換
    mean_preds_np = mean_preds.numpy()
    std_preds_np = std_preds.numpy()
    X_test_np = X_test.numpy()
    y_true_test = np.sin(2 * X_test_np.flatten())

    # Plotlyによる可視化
    fig = go.Figure()

    # 訓練データ点
    fig.add_trace(go.Scatter(
        x=x_train,
        y=y_train,
        mode='markers',
        name='Training Data',
        marker=dict(color='blue', size=6)
    ))

    # アウトライアを強調表示
    fig.add_trace(go.Scatter(
        x=x_train[outlier_indices],
        y=y_train[outlier_indices],
        mode='markers',
        name='Outliers',
        marker=dict(color='red', size=8, symbol='x')
    ))

    # 真の関数の軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=y_true_test,
        mode='lines',
        name='True Function',
        line=dict(color='green', width=2)
    ))

    # NN の予測の平均軌跡
    fig.add_trace(go.Scatter(
        x=X_test_np.flatten(),
        y=mean_preds_np.flatten(),
        mode='lines',
        name='NN Prediction (Mean)',
        line=dict(color='red', width=2)
    ))

    # 標準偏差の範囲をプロット (不確実性を表示)
    fig.add_trace(go.Scatter(
        x=np.concatenate([X_test_np.flatten(), X_test_np.flatten()[::-1]]),
        y=np.concatenate([mean_preds_np.flatten() - std_preds_np.flatten(), 
                          (mean_preds_np.flatten() + std_preds_np.flatten())[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.2)',  # 塗りつぶしの色
        line=dict(color='rgba(255, 0, 0, 0)'),  # 標準偏差の範囲の線の色を透明に
        hoverinfo="skip",
        showlegend=False
    ))

    # グラフのレイアウト設定
    fig.update_layout(
        title="BNN with Laplace Approximation (with Outliers)",
        xaxis_title="X",
        yaxis_title="y",
        width=800,
        height=600
    )

    fig.show()

if __name__ == "__main__":
    # データ生成
    x_train, y_train, y_true, outlier_indices = generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05)

    # モデルの設定
    model_settings = {
        "input_dim": 1,
        "output_dim": 1,
        "hidden_units": [10, 20],
        "activations": ["relu", "tanh"]
    }

    # トレーニングとラプラス近似での予測
    X_train, y_train_tensor, X_test, mean_preds, std_preds = train_and_predict_with_laplace(
        model_settings, x_train, y_train, l2_coeff=1e-2)

    # 結果の可視化
    plot_results(x_train, y_train, y_true, outlier_indices, X_test, mean_preds, std_preds)


Epoch [100/1000], Loss: 1.6473
Epoch [200/1000], Loss: 1.6399
Epoch [300/1000], Loss: 1.6072
Epoch [400/1000], Loss: 1.5904
Epoch [500/1000], Loss: 1.5883
Epoch [600/1000], Loss: 1.5878
Epoch [700/1000], Loss: 1.4482
Epoch [800/1000], Loss: 1.4038
Epoch [900/1000], Loss: 1.3567
Epoch [1000/1000], Loss: 1.3120
